In [25]:
import subprocess
import sys
import os
import time
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
from tqdm import tqdm

In [321]:
cpp_executable = "./gen"  # путь к скомпилированной C++ программе
output_file = "result.txt"  # файл, в который C++ программа пишет результат
reps = 100;
n = 100000;
delta = 0.01;
distrib = 1; # 0 - uni, 1 - exp, 2 - norm
click_size = 3;

input_args = list(map(str, [n, delta, reps, click_size, distrib]))
command = [cpp_executable] + input_args

In [322]:
res = list()
for i in range(5):
    process = subprocess.Popen(
        command,
        text=True,
        stdout=subprocess.PIPE
    )
    with tqdm(total=100, unit="%") as pbar:
        while True:
            line = process.stdout.readline()
            if not line:
                break
            try:
                progress_str = line.split("Progress:")[0].strip()
                progress_value = float(progress_str)
                pbar.update(progress_value - pbar.n) # Update by the difference
            except ValueError:
                pass
    with open(output_file, 'r', encoding='utf-8') as f:
        result = f.read()
    result = list(map(float, result.split()))
    E = sum(result) / reps
    sq = sum(num * num for num in result)
    D = sq/reps - E*E
    res.append([E, D])

100%|████████████████████████████████████████████████████████████████████████████| 100.0/100 [00:02<00:00, 36.86%/s]
100%|████████████████████████████████████████████████████████████████████████████| 100.0/100 [00:02<00:00, 35.50%/s]
100%|████████████████████████████████████████████████████████████████████████████| 100.0/100 [00:02<00:00, 36.74%/s]
100%|████████████████████████████████████████████████████████████████████████████| 100.0/100 [00:02<00:00, 37.72%/s]
100%|████████████████████████████████████████████████████████████████████████████| 100.0/100 [00:02<00:00, 35.99%/s]


In [323]:
match distrib:
    case 0: # Uniform
        P = 3 * delta**2
        P_ = 14/3 * delta**3 * 2
        P__ = 9 * delta**4 * 2
    case 1: # Exponential
        P = delta**2
        P_ = 7/6 * delta**3 * 2
        P__ = 9/5 * delta**4 * 2
    case 2: # Normal
        P = 0.276 * delta**2
        P_ = 0
        P__ = 0

In [324]:
E = n*(n-1)*(n-2)/6 * P
D = n*(n-1)*(n-2)/6 * P + n*(n-1)*(n-2)*(n-3)/4 * P_ + n*(n-1)*(n-2)*(n-3)*(n-4)/8 * P__ \
    + n*(n-1)*(n-2)*(n-3)*(n-4)*(n-5)/36 * P * P - n*(n-1)*(n-2)*n*(n-1)*(n-2)/36 * P * P
print(round(E), round(D))
for i in res:
    print(round(E/i[0],3), round(D/i[1], 3))

16666166670 20055846597885952
1.009 0.945
1.011 1.135
1.01 0.956
1.009 1.097
1.008 1.045
